In [ ]:
# Databricks notebook source
# 05_intermediate — Silver facts -> residual access findings
# Contract: docs/contracts/poc-intermediate-gold-schema.md
# Eval is intentionally not read by this notebook.


In [ ]:
dbutils.widgets.text("catalog", "access_drift")
dbutils.widgets.text("run_id", "poc")

CATALOG = dbutils.widgets.get("catalog")
RUN_ID = dbutils.widgets.get("run_id") or "poc"

In [ ]:
from pyspark.sql import functions as F

principals = spark.table(f"{CATALOG}.silver.silver_principals")
credentials = spark.table(f"{CATALOG}.silver.silver_credentials")
assets = spark.table(f"{CATALOG}.silver.silver_assets")
edges = spark.table(f"{CATALOG}.silver.silver_edges")

owners = principals.where(
    (F.col("principal_kind") == "HI") &
    (F.col("status") == "terminated")
).alias("owner")

entra_nhis = principals.where(
    (F.col("principal_kind") == "NHI") &
    (F.col("principal_type") == "app_registration") &
    (F.col("status") == "active")
).alias("nhi")

github_nhis = principals.where(
    (F.col("principal_kind") == "NHI") &
    (F.col("principal_type") == "github_app") &
    (F.col("status") == "active")
).alias("nhi")

hi_subjects = principals.where(
    (F.col("principal_kind") == "HI") &
    (F.col("source_principal_id").startswith("gws-") | F.col("source_principal_id").startswith("slack-")) &
    (F.col("status").isin("terminated", "inactive"))
).alias("hi")

active_credentials = credentials.where(
    (F.col("status") == "active") &
    (F.col("is_active") == F.lit(True))
).alias("credential")

owner_edges = edges.where(
    (F.col("relationship") == "owns_principal") &
    (F.col("from_type") == "principal") &
    (F.col("to_type") == "principal") &
    (F.col("status") == "active")
).alias("owner_edge")

access_edges = edges.where(
    (F.col("relationship") == "accesses_asset") &
    (F.col("from_type") == "principal") &
    (F.col("to_type") == "asset") &
    (F.col("status") == "active")
).alias("access_edge")

asset_nodes = assets.alias("asset")


In [ ]:
entra_joined = (
    owner_edges
    .join(owners, F.col("owner_edge.from_id") == F.col("owner.principal_id"), "inner")
    .join(entra_nhis, F.col("owner_edge.to_id") == F.col("nhi.principal_id"), "inner")
    .join(active_credentials, F.col("credential.bound_principal_id") == F.col("nhi.principal_id"), "inner")
    .join(access_edges, F.col("access_edge.from_id") == F.col("nhi.principal_id"), "inner")
    .join(asset_nodes, F.col("access_edge.to_id") == F.col("asset.asset_id"), "inner")
)

github_joined = (
    owner_edges
    .join(owners, F.col("owner_edge.from_id") == F.col("owner.principal_id"), "inner")
    .join(github_nhis, F.col("owner_edge.to_id") == F.col("nhi.principal_id"), "inner")
    .join(active_credentials, F.col("credential.bound_principal_id") == F.col("nhi.principal_id"), "inner")
    .join(access_edges, F.col("access_edge.from_id") == F.col("nhi.principal_id"), "inner")
    .join(asset_nodes, F.col("access_edge.to_id") == F.col("asset.asset_id"), "inner")
)

hi_joined = (
    hi_subjects
    .join(access_edges, F.col("access_edge.from_id") == F.col("hi.principal_id"), "inner")
    .join(asset_nodes, F.col("access_edge.to_id") == F.col("asset.asset_id"), "inner")
)

entra_risk_case_key = F.concat_ws(
    ":", F.lit("nhi"), F.col("nhi.source_principal_id"), F.col("asset.source_asset_id"), F.lit("nhi_residual_access")
)
github_risk_case_key = F.concat_ws(
    ":", F.lit("nhi"), F.col("nhi.source_principal_id"), F.col("asset.source_asset_id"), F.lit("nhi_residual_access")
)
hi_risk_case_key = F.concat_ws(
    ":", F.lit("hi"), F.col("hi.source_principal_id"), F.col("asset.source_asset_id"), F.lit("hi_residual_access")
)

def nhi_finding_type_label(source_principal_id_col, source_asset_id_col, permission_col):
    return (
        F.when(source_asset_id_col == F.lit("asset-customer-export-storage-prod"), F.lit("NHI Access Drift"))
         .when(source_asset_id_col == F.lit("asset-customer-profile-export-0002"), F.lit("NHI Credential Drift"))
         .when(source_asset_id_col == F.lit("asset-finance-reporting-0003"), F.lit("NHI Permission Drift"))
         .when(source_principal_id_col == F.lit("gh-app-security-audit-bot"), F.lit("NHI Credential Drift"))
         .when(source_principal_id_col.isin("gh-app-data-export-bot", "gh-app-finance-ci-bot"), F.lit("NHI Permission Drift"))
         .when(permission_col.isin("write"), F.lit("NHI Permission Drift"))
         .otherwise(F.lit("NHI Access Drift"))
    )

def saas_source_system(source_principal_id_col):
    return (
        F.when(source_principal_id_col.startswith("gws-"), F.lit("google_workspace"))
         .when(source_principal_id_col.startswith("slack-"), F.lit("slack"))
         .otherwise(F.lit("identity_seed"))
    )

def hi_finding_type_label(source_principal_id_col):
    return (
        F.when(source_principal_id_col.isin("gws-user-oh.sehun", "slack-user-lim.taehyun", "slack-user-moon.jae"), F.lit("HI Stale Access"))
         .when(source_principal_id_col.isin("gws-user-park.jun", "gws-user-jung.woo", "slack-user-han.jimin"), F.lit("HI Permission Drift"))
         .otherwise(F.lit("HI Access Drift"))
    )

entra_findings = entra_joined.select(
    F.sha2(F.concat_ws(":", F.lit("finding"), entra_risk_case_key), 256).alias("finding_id"),
    entra_risk_case_key.alias("risk_case_key"),
    F.lit("nhi_residual_access").alias("drift_type"),
    F.lit("nhi").alias("subject_type"),
    F.col("nhi.principal_kind").alias("principal_kind"),
    F.col("nhi.source_system").alias("source_system"),
    F.col("nhi.principal_type").alias("principal_type"),
    nhi_finding_type_label(F.col("nhi.source_principal_id"), F.col("asset.source_asset_id"), F.col("access_edge.permission")).alias("finding_type_label"),
    F.array(F.lit("terminated_owner"), F.lit("active_credential"), F.lit("active_asset_access")).alias("risk_factors"),
    F.col("owner.principal_id").alias("owner_principal_id"),
    F.col("owner.source_principal_id").alias("owner_source_principal_id"),
    F.col("nhi.principal_id").alias("subject_principal_id"),
    F.col("nhi.source_principal_id").alias("subject_source_principal_id"),
    F.col("credential.credential_id").alias("credential_id"),
    F.col("asset.asset_id").alias("asset_id"),
    F.col("asset.source_asset_id").alias("asset_source_id"),
    F.col("owner_edge.edge_id").alias("owner_edge_id"),
    F.col("access_edge.edge_id").alias("access_edge_id"),
    F.col("access_edge.permission").alias("permission"),
    F.col("owner.status").alias("owner_status"),
    F.col("credential.is_active").alias("credential_is_active"),
    F.col("asset.environment").alias("asset_environment"),
    F.col("asset.sensitivity").alias("asset_sensitivity"),
    F.current_timestamp().alias("detected_at"),
    F.lit(RUN_ID).alias("run_id"),
)

github_findings = github_joined.select(
    F.sha2(F.concat_ws(":", F.lit("finding"), github_risk_case_key), 256).alias("finding_id"),
    github_risk_case_key.alias("risk_case_key"),
    F.lit("nhi_residual_access").alias("drift_type"),
    F.lit("nhi").alias("subject_type"),
    F.col("nhi.principal_kind").alias("principal_kind"),
    F.lit("github").alias("source_system"),
    F.col("nhi.principal_type").alias("principal_type"),
    nhi_finding_type_label(F.col("nhi.source_principal_id"), F.col("asset.source_asset_id"), F.col("access_edge.permission")).alias("finding_type_label"),
    F.array(F.lit("terminated_owner"), F.lit("active_credential"), F.lit("active_asset_access"), F.lit("github_app")).alias("risk_factors"),
    F.col("owner.principal_id").alias("owner_principal_id"),
    F.col("owner.source_principal_id").alias("owner_source_principal_id"),
    F.col("nhi.principal_id").alias("subject_principal_id"),
    F.col("nhi.source_principal_id").alias("subject_source_principal_id"),
    F.col("credential.credential_id").alias("credential_id"),
    F.col("asset.asset_id").alias("asset_id"),
    F.col("asset.source_asset_id").alias("asset_source_id"),
    F.col("owner_edge.edge_id").alias("owner_edge_id"),
    F.col("access_edge.edge_id").alias("access_edge_id"),
    F.col("access_edge.permission").alias("permission"),
    F.col("owner.status").alias("owner_status"),
    F.col("credential.is_active").alias("credential_is_active"),
    F.col("asset.environment").alias("asset_environment"),
    F.col("asset.sensitivity").alias("asset_sensitivity"),
    F.current_timestamp().alias("detected_at"),
    F.lit(RUN_ID).alias("run_id"),
)

hi_findings = hi_joined.select(
    F.sha2(F.concat_ws(":", F.lit("finding"), hi_risk_case_key), 256).alias("finding_id"),
    hi_risk_case_key.alias("risk_case_key"),
    F.lit("hi_residual_access").alias("drift_type"),
    F.lit("hi").alias("subject_type"),
    F.col("hi.principal_kind").alias("principal_kind"),
    saas_source_system(F.col("hi.source_principal_id")).alias("source_system"),
    F.col("hi.principal_type").alias("principal_type"),
    hi_finding_type_label(F.col("hi.source_principal_id")).alias("finding_type_label"),
    F.array(F.lit("inactive_or_terminated_subject"), F.lit("active_asset_access"), F.lit("saas_hi")).alias("risk_factors"),
    F.col("hi.principal_id").alias("owner_principal_id"),
    F.col("hi.source_principal_id").alias("owner_source_principal_id"),
    F.col("hi.principal_id").alias("subject_principal_id"),
    F.col("hi.source_principal_id").alias("subject_source_principal_id"),
    F.lit(None).cast("string").alias("credential_id"),
    F.col("asset.asset_id").alias("asset_id"),
    F.col("asset.source_asset_id").alias("asset_source_id"),
    F.lit(None).cast("string").alias("owner_edge_id"),
    F.col("access_edge.edge_id").alias("access_edge_id"),
    F.col("access_edge.permission").alias("permission"),
    F.col("hi.status").alias("owner_status"),
    F.lit(None).cast("boolean").alias("credential_is_active"),
    F.col("asset.environment").alias("asset_environment"),
    F.col("asset.sensitivity").alias("asset_sensitivity"),
    F.current_timestamp().alias("detected_at"),
    F.lit(RUN_ID).alias("run_id"),
)

findings = entra_findings.unionByName(github_findings).unionByName(hi_findings)


In [ ]:
target = f"{CATALOG}.intermediate.nhi_residual_access_findings"

classification_columns = {
    "principal_kind": "STRING",
    "source_system": "STRING",
    "principal_type": "STRING",
    "finding_type_label": "STRING",
    "risk_factors": "ARRAY<STRING>",
}
existing_columns = set(spark.table(target).columns)
missing_classification_columns = [
    column for column in classification_columns
    if column not in existing_columns
]
if missing_classification_columns:
    add_columns_sql = ", ".join(
        f"{column} {classification_columns[column]}"
        for column in missing_classification_columns
    )
    spark.sql(f"ALTER TABLE {target} ADD COLUMNS ({add_columns_sql})")
    print(f"Added classification columns to {target}: {missing_classification_columns}")

nullable_evidence_columns = ["credential_id", "owner_edge_id", "credential_is_active", "owner_status"]
for column in nullable_evidence_columns:
    try:
        spark.sql(f"ALTER TABLE {target} ALTER COLUMN {column} DROP NOT NULL")
    except Exception as exc:
        message = str(exc).lower()
        if "not null" not in message and "nullable" not in message:
            raise

target_columns = spark.table(target).columns
finding_columns = findings.columns

missing_columns = [column for column in target_columns if column not in finding_columns]
extra_columns = [column for column in finding_columns if column not in target_columns]
if missing_columns or extra_columns:
    raise ValueError(
        f"schema mismatch for {target}: missing={missing_columns}, extra={extra_columns}"
    )

if finding_columns != target_columns:
    print(f"Aligning {target} columns to target table order")
    print(f"dataframe columns: {finding_columns}")
    print(f"target columns: {target_columns}")

findings_aligned = findings.select(*target_columns)
findings_aligned.write.format("delta").mode("overwrite").saveAsTable(target)

result = spark.table(target)
count = result.count()
assert count >= 1, "expected at least one residual access finding"

duplicate_keys = result.groupBy("risk_case_key").count().where("count > 1").count()
assert duplicate_keys == 0, f"duplicate risk_case_key rows: {duplicate_keys}"

hash_style_keys = result.where("risk_case_key RLIKE '^(nhi|hi):[0-9a-f]{64}$'").count()
assert hash_style_keys == 0, f"hash-style risk_case_key rows remain: {hash_style_keys}"

invalid_contract_keys = result.where(
    "risk_case_key <> concat_ws(':', subject_type, subject_source_principal_id, asset_source_id, drift_type)"
).count()
assert invalid_contract_keys == 0, f"risk_case_key contract mismatches: {invalid_contract_keys}"

required = [
    "finding_id", "risk_case_key", "principal_kind", "source_system",
    "principal_type", "finding_type_label", "risk_factors",
    "owner_principal_id", "subject_principal_id",
    "asset_id", "access_edge_id",
]
missing_required = result.where(
    F.expr(" OR ".join(f"{column} IS NULL" for column in required))
).count()
assert missing_required == 0, f"rows with missing required evidence: {missing_required}"

nhi_missing_evidence = result.where(
    "principal_kind = 'NHI' AND (credential_id IS NULL OR credential_is_active IS NULL)"
).count()
assert nhi_missing_evidence == 0, f"NHI rows with missing credential/owner evidence: {nhi_missing_evidence}"

nhi_missing_owner_evidence = result.where(
    "principal_kind = 'NHI' AND (owner_edge_id IS NULL OR owner_status IS NULL)"
).count()
assert nhi_missing_owner_evidence == 0, f"NHI rows with missing owner edge/status evidence: {nhi_missing_owner_evidence}"

source_distribution = result.groupBy("source_system").count().orderBy("source_system")
label_distribution = result.groupBy("finding_type_label").count().orderBy("finding_type_label")
cross_distribution = result.groupBy("finding_type_label", "source_system", "principal_type").count().orderBy(
    "finding_type_label", "source_system", "principal_type"
)
print("source_system distribution")
source_distribution.show(truncate=False)
print("finding_type_label distribution")
label_distribution.show(truncate=False)
print("finding_type_label/source_system/principal_type distribution")
cross_distribution.show(truncate=False)

credential_label_count = result.where("finding_type_label = 'NHI Credential Drift'").count()
assert credential_label_count < count, "all findings were labeled NHI Credential Drift"

expected_sources = {"github", "google_workspace", "slack"}
actual_sources = {row["source_system"] for row in result.select("source_system").distinct().collect()}
missing_sources = expected_sources - actual_sources
assert not missing_sources, f"missing expected SaaS source_system values: {sorted(missing_sources)}; actual={sorted(actual_sources)}"

expected_labels = {
    "NHI Access Drift",
    "NHI Credential Drift",
    "NHI Permission Drift",
    "HI Access Drift",
    "HI Permission Drift",
    "HI Stale Access",
}
actual_labels = {row["finding_type_label"] for row in result.select("finding_type_label").distinct().collect()}
missing_labels = expected_labels - actual_labels
assert not missing_labels, f"missing expected finding_type_label values: {sorted(missing_labels)}; actual={sorted(actual_labels)}"

result.orderBy("subject_source_principal_id", "asset_source_id").show(truncate=False)
print(f"intermediate findings: {count} rows")
print("intermediate smoke test passed")
